In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [6]:
from graphviz import Digraph

def trace(root):
    #builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()
    def build(v):
        if node not in nodes:
            nodes.add(node)
            for parent in node._prev:
                edges.add((parent, node))
                build(parent)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = Left to Right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        #for any value in the graph, create a rectangular ('record') node for it
        dot.node(name = uid, label = "{ data %.4f}" % (n.data, ), shape = 'record')
        if n._op:
            # if this value is a result of an operation, create as operation node for it
            dot.node(name = uid + n._op, label = n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the operation node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [3]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other_value):
        out = Value(self.data + other_value.data, (self, other_value), '+')
        return out

    def __sub__(self, other_value):
        out = Value(self.data - other_value.data, (self, other_value), '-')
        return out

    def __mul__(self, other_value):
        out = Value(self.data * other_value.data, (self, other_value), '*')
        return out

    def __truediv__(self, other_value):
        out = Value(self.data / other_value.data, (self, other_value), '/')
        return out

In [4]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)

# (a.__mul__(b)).__add__(c)
d = a*b + c
d

Value(data=4.0)